# Laboratory 3, Part A - Text to numbers with Bag-of-Words

**Course:** Introduction to Artificial Intelligence (AI-101L)   **Student:** Arham Rizwan

Install once if needed: `pip install scikit-learn Pillow numpy pandas matplotlib`

Machine learning models need numeric vectors, not sentences. Bag-of-Words counts how often each
word occurs in a document and ignores grammar and word order.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "The sun is shining today.",
    "The weather is good, the sun is great.",
    "A sunny day is a wonderful day.",
]

for number, text in enumerate(documents, start=1):
    print(f"Doc {number}: {text}")

Doc 1: The sun is shining today.
Doc 2: The weather is good, the sun is great.
Doc 3: A sunny day is a wonderful day.


## Task 3.1 - Vectorise with `CountVectorizer`

In [2]:
vectorizer = CountVectorizer()
sparse_counts = vectorizer.fit_transform(documents)

vocabulary = vectorizer.get_feature_names_out()
counts = sparse_counts.toarray()

print("Vocabulary:", list(vocabulary))
print("Matrix shape (documents x terms):", counts.shape)

Vocabulary: ['day', 'good', 'great', 'is', 'shining', 'sun', 'sunny', 'the', 'today', 'weather', 'wonderful']
Matrix shape (documents x terms): (3, 11)


In [3]:
table = pd.DataFrame(counts, columns=vocabulary, index=[f"Doc {i}" for i in range(1, len(documents) + 1)])
table

,day,good,great,is,shining,sun,sunny,the,today,weather,wonderful
Doc 1,0,0,0,1,1,1,0,1,1,0,0
Doc 2,0,1,1,2,0,1,0,2,0,1,0
Doc 3,2,0,0,1,0,0,1,0,0,0,1


### Two silent defaults worth checking

In [4]:
print("'the' in vocabulary:", "the" in vocabulary, "(lowercasing merges 'The' and 'the')")
print("'a' in vocabulary  :", "a" in vocabulary, "(single-character tokens are dropped)")
print("lowercase setting  :", vectorizer.lowercase)
print("token pattern      :", vectorizer.token_pattern)

'the' in vocabulary: True (lowercasing merges 'The' and 'the')
'a' in vocabulary  : False (single-character tokens are dropped)
lowercase setting  : True
token pattern      : (?u)\b\w\w+\b


In [5]:
zeros = int((counts == 0).sum())
print(f"Zero cells: {zeros} of {counts.size}  ->  sparsity {zeros / counts.size:.1%}")
print("fit_transform returned:", type(sparse_counts).__name__, "(a sparse matrix)")

Zero cells: 18 of 33  ->  sparsity 54.5%
fit_transform returned: csr_matrix (a sparse matrix)


## Discussion questions (Part A)

**1. What does each column represent?**

Each column is one unique word (term) from the vocabulary learned across the whole corpus. The number in a cell says how many times that word appears in that row's document. Here there are 11 columns, so 11 features.

**2. What is the machine-understandable format here?**

The machine-understandable format is the 3 x 11 matrix of integer word counts: every document becomes a fixed-length numeric vector (length 11), and any algorithm that accepts numeric vectors can now work with text. scikit-learn stores it as a sparse matrix.

**3. How is it an oversimplification?**

BoW throws away a lot. Word order is gone, so 'dog bites man' and 'man bites dog' get the same vector. Grammar, context and meaning are gone too: 'sun' and 'sunny' become two unrelated columns, and synonyms are not linked. The defaults add more loss: everything is lowercased, so 'The' and 'the' merge, and single-character tokens such as 'a' are dropped. Raw counts also favour long documents, and any word not in the training vocabulary is ignored later.

## In-lab exercises

**Exercise 1.** Re-run with `ngram_range=(1, 2)` and compare vocabulary sizes.

In [6]:
unigrams = CountVectorizer(ngram_range=(1, 1)).fit(documents)
uni_and_bigrams = CountVectorizer(ngram_range=(1, 2)).fit(documents)

n1 = len(unigrams.get_feature_names_out())
n2 = len(uni_and_bigrams.get_feature_names_out())
print("ngram_range=(1,1):", n1, "features")
print("ngram_range=(1,2):", n2, "features  (x", round(n2 / n1, 1), ")")
print()
print("Bigrams added:", [t for t in uni_and_bigrams.get_feature_names_out() if " " in t])

ngram_range=(1,1): 11 features
ngram_range=(1,2): 24 features  (x 2.2 )

Bigrams added: ['day is', 'good the', 'is good', 'is great', 'is shining', 'is wonderful', 'shining today', 'sun is', 'sunny day', 'the sun', 'the weather', 'weather is', 'wonderful day']


A document with *n* tokens contains *n - 1* adjacent word pairs, and most of those pairs are
unique to one document, so the vocabulary grows roughly with the total amount of text instead of the
number of distinct words. Bigrams recover a little word-order information, at the cost of a wider,
sparser matrix.

**Exercise 2.** Add a fourth document that repeats *sun* five times and compare.

In [7]:
documents_4 = documents + ["sun sun sun sun sun and more sun today"]

vec4 = CountVectorizer()
counts4 = vec4.fit_transform(documents_4).toarray()
table4 = pd.DataFrame(counts4, columns=vec4.get_feature_names_out(),
                      index=[f"Doc {i}" for i in range(1, len(documents_4) + 1)])
display(table4)

print("Words per document (row sums):")
print(table4.sum(axis=1).to_string())
print()
print("Share of each document that is the word 'sun':")
print((table4["sun"] / table4.sum(axis=1)).round(2).to_string())

,and,day,good,great,is,more,shining,sun,sunny,the,today,weather,wonderful
Doc 1,0,0,0,0,1,0,1,1,0,1,1,0,0
Doc 2,0,0,1,1,2,0,0,1,0,2,0,1,0
Doc 3,0,2,0,0,1,0,0,0,1,0,0,0,1
Doc 4,1,0,0,0,0,1,0,6,0,0,1,0,0


Words per document (row sums):
Doc 1    5
Doc 2    8
Doc 3    5
Doc 4    9

Share of each document that is the word 'sun':
Doc 1    0.20
Doc 2    0.12
Doc 3    0.00
Doc 4    0.67


Doc 4 has by far the largest 'sun' count simply because it repeats the word and is longer, not
because it says anything more about the sun. Raw counts therefore favour long and repetitive
documents. Dividing each row by its length (the second table) or using TF-IDF corrects part of
this.